# 4. Logistic Regression

Khác với Regression thông thường dự đoán giá trị liên tục, Logistic Regression dự đoán **xác suất (probability)** của một mẫu thuộc về một lớp bằng cách sử dụng hàm sigmoid (đưa giá trị về khoảng 0-1). Nếu xác suất > decision threshold (thường là 0.5), nhãn sẽ là 1.

## Hyperparameters
- `C`: Nghịch đảo của sức mạnh tinh chỉnh (inverse of regularization strength). C càng nhỏ, regularization càng mạnh (chống overfitting).
- `solver`: Thuật toán dùng để tối ưu hóa (vd: `lbfgs`, `liblinear`).

In [ ]:
import sys
sys.path.append("..")
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from src.preprocessing import get_preprocessing_pipeline
from src.models.logistic_regression import get_logistic_regression
from src.metrics import calculate_metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

df = pd.read_csv("../data/heart_cleveland_upload.csv")
X = df.drop("condition", axis=1)
y = df["condition"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
pipeline = Pipeline([
    ("preprocessor", get_preprocessing_pipeline()),
    ("model", get_logistic_regression())
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["lbfgs", "liblinear"]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

## Đánh giá mô hình

In [ ]:
y_pred = grid.predict(X_test)
y_score = grid.predict_proba(X_test)[:, 1]
metrics = calculate_metrics(y_test, y_pred, y_score)
print(metrics)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.show()